# AI-Based Behavioral IDS - Data Exploration

This notebook explores the UNSW-NB15 and CICIDS2017 datasets used for training the Intrusion Detection System.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.data_loader import DataLoader
from data.preprocessor import DataPreprocessor
from data.smote_balancing import check_class_imbalance

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully!")

## 1. Load Dataset

We can use either the real datasets (UNSW-NB15, CICIDS2017) or generate synthetic data for demonstration.

In [ ]:
# Option 1: Generate synthetic data
df = DataLoader.generate_synthetic_data(n_samples=10000, n_features=47)

# Option 2: Load UNSW-NB15 (uncomment if dataset is available)
# loader = DataLoader()
# df = loader.load_unsw_nb15()

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns[:10])}...")
print(f"\nFirst few rows:")
df.head()

## 2. Dataset Summary

In [ ]:
# Dataset info
loader = DataLoader()
loader.print_dataset_summary(df, "Demo Dataset")

# Basic statistics
df.describe()

## 3. Class Distribution Analysis

In [ ]:
# Check class imbalance
is_imbalanced, info = check_class_imbalance(df['label'].values)

print(f"Class Imbalance Analysis:")
print(f"  Is imbalanced: {is_imbalanced}")
print(f"  Imbalance ratio: {info['imbalance_ratio']:.4f}")
print(f"  Majority: {info['majority_count']} ({info['majority_percentage']:.1f}%)")
print(f"  Minority: {info['minority_count']} ({info['minority_percentage']:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary labels
label_counts = df['label'].value_counts()
axes[0].bar(['Normal', 'Attack'], [label_counts[0], label_counts[1]], 
          color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Binary Label Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([label_counts[0], label_counts[1]]):
    axes[0].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

# Attack categories
if 'attack_cat' in df.columns:
    attack_counts = df['attack_cat'].value_counts()
    axes[1].barh(range(len(attack_counts)), attack_counts.values, color='coral')
    axes[1].set_yticks(range(len(attack_counts)))
    axes[1].set_yticklabels(attack_counts.index)
    axes[1].set_title('Attack Category Distribution', fontweight='bold')
    axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 4. Feature Distribution Analysis

In [ ]:
# Select numerical features
num_features = df.select_dtypes(include=[np.number]).columns[:5]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    if i < len(axes):
        axes[i].hist(df[df['label']==0][col], bins=30, alpha=0.6, 
                    label='Normal', color='steelblue', density=True)
        axes[i].hist(df[df['label']==1][col], bins=30, alpha=0.6, 
                    label='Attack', color='coral', density=True)
        axes[i].set_title(f'{col}', fontweight='bold')
        axes[i].legend()

plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation matrix
corr_cols = df.select_dtypes(include=[np.number]).columns[:10]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
          fmt='.2f', square=True)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Data Preprocessing Demo

In [ ]:
# Preprocess data
preprocessor = DataPreprocessor(
    normalization_method='minmax',
    sequence_length=100,
    use_pca=True,
    n_components=10
)

results = preprocessor.fit_transform(df, test_size=0.2)

print("Preprocessing Results:")
print(f"  Train sequences: {results['X_train_seq'].shape}")
print(f"  Test sequences: {results['X_test_seq'].shape}")
print(f"  Train flat: {results['X_train_flat'].shape}")
print(f"  Test flat: {results['X_test_flat'].shape}")
print(f"  Number of features: {results['n_features']}")
print(f"\nClass distribution:")
print(f"  Train: {results['class_distribution']['train']}")
print(f"  Test: {results['class_distribution']['test']}")

## Summary

This notebook demonstrated:
1. Dataset loading and structure analysis
2. Class distribution and imbalance detection
3. Feature distribution visualization
4. Correlation analysis
5. Complete preprocessing pipeline

Next: See `02_Model_Training.ipynb` for model training.